<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/rag_token_train_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers "datasets<3" sentencepiece accelerate faiss-cpu

In [ ]:
import re
import torch
import faiss
import numpy as np
import pandas as pd

from datasets import load_dataset, Dataset
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

# Load NQ dataset and split for train/eval

In [ ]:
nq = load_dataset("sentence-transformers/natural-questions", split="train[:2000]")
split_dataset = nq.train_test_split(test_size=0.2, seed=42)
#tiny_split = nq.train_test_split(test_size=0.01, seed=42)
train_dataset = split_dataset["train"].select(range(1000))
eval_dataset = split_dataset["test"].select(range(200))
dataset_name = "natural-questions"

# Helper functions

In [ ]:
def get_question(example):
  return example["query"]

def shorten_answer(answer):
  answer = str(answer).strip()
  words = answer.split()
  answer = " ".join(words[:10])
  return answer

def get_answer(example):
  answer = example["answer"]
  if isinstance(answer, list):
    answer = answer[0] if len(answer)>0 else ""
  answer = shorten_answer(answer)
  return answer

# Token-level F1

In [ ]:
def qa_f1(prediction, gold):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = set(pred_tokens) & set(gold_tokens)
    num_same = sum(
        min(pred_tokens.count(tok), gold_tokens.count(tok))
        for tok in common
    )

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

# Normalization function

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Retrieval Models

In [ ]:
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = question_encoder.to(device)

In [ ]:
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = context_encoder.to(device)

In [ ]:
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
bart_model = bart_model.to(device)

# Question encoding

In [ ]:
def encode_question(question):
  inputs = question_tokenizer(question, return_tensors="pt", truncation=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = question_encoder(**inputs).pooler_output
  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Passage encoding

In [ ]:
def encode_passage(passage):
  inputs = context_tokenizer(passage, return_tensors="pt", truncation=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = context_encoder(**inputs).pooler_output

  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Token-style retrieval

In [ ]:
def build_doc_input(question, relevant_passages):
  input = []
  for passage in relevant_passages:
    input.append(f"{passage} question: {question}")
  return input

# Building FAISS index

In [ ]:
def build_faiss_index(passages):
  passages_embeddings = np.stack([encode_passage(passage) for passage in passages]).astype(np.float32)
  faiss.normalize_L2(passages_embeddings)
  index = faiss.IndexFlatIP(passages_embeddings.shape[1])
  index.add(passages_embeddings)
  return index

# Building retrieval corpus for a dataset

In [ ]:
def retrieval_top_k(question, index, passages, k = 10):
  question_embedding = encode_question(question)
  question_embedding = question_embedding.reshape(1, -1)
  faiss.normalize_L2(question_embedding)
  distances, indices = index.search(question_embedding, k)
  relevant_passages = [passages[i] for i in indices[0]]
  return relevant_passages, distances[0]

In [ ]:
wiki_like = load_dataset("squad", split="train[:5000]")

def build_passage_corpus_from_contexts(dataset):
    passages = []
    for i in range(len(dataset)):
        context = str(dataset[i]["context"]).strip()
        if context != "":
            passages.append(context)

    passages = list(dict.fromkeys(passages))
    return passages

passages = build_passage_corpus_from_contexts(wiki_like)
index = build_faiss_index(passages)

In [ ]:
def rag_token_generate(question, passages, index, model, tokenizer, k=10, max_new_tokens=32):
    model.eval()

    # 1. Retrieve top-k documents
    relevant_passages, scores = retrieval_top_k(question, index, passages, k=k)

    scores = torch.tensor(scores, dtype=torch.float32, device=device)
    log_retriever_probs = F.log_softmax(scores, dim=-1)  # shape: [k]

    # 2. Build one input per document
    doc_inputs = [
        f"{passage} question: {question}"
        for passage in relevant_passages
    ]

    inputs = tokenizer(
        doc_inputs,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # 3. Start decoder with BART start token
    decoder_input_ids = torch.full(
        (k, 1),
        model.config.decoder_start_token_id,
        dtype=torch.long,
        device=device
    )

    generated_ids = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                decoder_input_ids=decoder_input_ids
            )

            # logits for next token from each document
            next_token_logits = outputs.logits[:, -1, :]  # shape: [k, vocab_size]

            # BART log probs for next token under each document
            next_token_log_probs = F.log_softmax(next_token_logits, dim=-1)

            # Add retriever log probability to each document's distribution
            weighted_log_probs = next_token_log_probs + log_retriever_probs.unsqueeze(1)

            # RAG-Token marginalization over documents
            marginal_log_probs = torch.logsumexp(weighted_log_probs, dim=0)  # shape: [vocab_size]

            # Greedy choose best next token
            next_token_id = torch.argmax(marginal_log_probs).unsqueeze(0)

            # Stop if EOS
            if next_token_id.item() == tokenizer.eos_token_id:
                break

            generated_ids.append(next_token_id.item())

            # Add same generated token to decoder input for every document
            next_token_for_all_docs = next_token_id.repeat(k, 1)
            decoder_input_ids = torch.cat(
                [decoder_input_ids, next_token_for_all_docs],
                dim=1
            )

    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return answer, relevant_passages

# Custom Loss function

In [ ]:
import torch.nn.functional as F

def rag_token_loss(question, answer, passages, index, model, tokenizer, k=10):
    # 1. Retrieve K documents
    relevant_passages, scores = retrieval_top_k(question, index, passages, k=k)

    # 2. Convert retriever scores to log probabilities
    scores = torch.tensor(scores, dtype=torch.float32, device=device)
    log_retriever_probs = F.log_softmax(scores, dim=-1)  # shape: [k]

    # 3. Build one BART input per retrieved document
    doc_inputs = [
        f"{passage} question: {question}"
        for passage in relevant_passages
    ]

    # 4. Tokenize document-question inputs
    inputs = tokenizer(
        doc_inputs,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # 5. Tokenize target answer
    target = tokenizer(
        answer,
        return_tensors="pt",
        truncation=True,
        max_length=32
    )
    labels = target["input_ids"].to(device)  # shape: [1, target_len]

    # Remove final token from decoder input, predict next tokens
    decoder_input_ids = labels[:, :-1]
    target_ids = labels[:, 1:]

    # Repeat decoder inputs for each retrieved document
    decoder_input_ids = decoder_input_ids.repeat(k, 1)  # shape: [k, target_len - 1]

    # 6. Run BART once for all K documents
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        decoder_input_ids=decoder_input_ids
    )

    # logits shape: [k, target_len - 1, vocab_size]
    logits = outputs.logits

    # 7. Convert BART logits to log probabilities
    token_log_probs = F.log_softmax(logits, dim=-1)

    # 8. Get log prob of the gold target token under each document
    target_ids = target_ids.repeat(k, 1)  # shape: [k, target_len - 1]

    gold_token_log_probs = token_log_probs.gather(
        dim=-1,
        index=target_ids.unsqueeze(-1)
    ).squeeze(-1)  # shape: [k, target_len - 1]

    # 9. Add retriever log probability to every token position
    weighted_log_probs = gold_token_log_probs + log_retriever_probs.unsqueeze(1)

    # 10. Marginalize over documents for EACH TOKEN
    marginal_token_log_probs = torch.logsumexp(
        weighted_log_probs,
        dim=0
    )  # shape: [target_len - 1]

    # 11. Negative marginal log likelihood
    loss = -marginal_token_log_probs.mean()

    return loss

In [ ]:
model = bart_model
tokenizer = bart_tokenizer

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

model.train()

num_epochs = 3
k = 10

for epoch in range(num_epochs):
    total_loss = 0.0

    for i in range(len(train_dataset)):
        example = train_dataset[i]

        question = get_question(example)
        answer = get_answer(example)

        if str(answer).strip() == "":
            continue

        optimizer.zero_grad()

        loss = rag_token_loss(
            question=question,
            answer=answer,
            passages=passages,
            index=index,
            model=bart_model,
            tokenizer=bart_tokenizer,
            k=k
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 20 == 0:
            print(f"Epoch {epoch + 1}, step {i}, loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_dataset)
    print(f"Epoch {epoch + 1} average loss: {avg_loss:.4f}")

In [ ]:
results = []

for i in range(len(eval_dataset)):
    example = eval_dataset[i]

    question = get_question(example)
    gold = get_answer(example)

    pred, docs = rag_token_generate(
        question=question,
        passages=passages,
        index=index,
        model=bart_model,
        tokenizer=bart_tokenizer,
        k=10
    )

    f1 = qa_f1(pred, gold)

    results.append({
        "question": question,
        "gold_answer": gold,
        "prediction": pred,
        "retrieved_docs": docs,
        "f1": f1
    })

results_df = pd.DataFrame(results)
results_df.to_csv("rag_token_results.csv", index=False)

print("Saved rag_token_results.csv")
print("Final F1:", results_df["f1"].mean() * 100)

for i in range(5):
    print("QUESTION:", results_df["question"][i])
    print("GOLD:", results_df["gold_answer"][i])
    print("PRED:", results_df["prediction"][i])
    print("RETRIEVED:", results_df["retrieved_docs"][i])
    print("-" * 80)